# NSL-KDD Stacking Model
Upload the merged `NSL-KDD_merged.csv`, train the proposed stacking model, evaluate it, and download a reusable PKL model bundle.

## 1. Install dependencies

In [ ]:
!pip -q install numpy pandas scikit-learn statsmodels

## 2. Upload the merged CSV
Run this cell and click **Choose Files**. Select `NSL-KDD_merged.csv` from your computer.

In [ ]:
from google.colab import files

uploaded = files.upload()
if not uploaded:
    raise RuntimeError('No CSV file was uploaded.')

csv_name = next(iter(uploaded))
print(f'Using uploaded file: {csv_name}')

## 3. Load and prepare the data

In [ ]:
import pickle
import time

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import BernoulliNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, RobustScaler
from sklearn.tree import DecisionTreeClassifier

df = pd.read_csv(csv_name)
required = {'class', 'difficulty_level', 'source_file', 'num_outbound_cmds',
            'protocol_type', 'service', 'flag'}
missing = required - set(df.columns)
if missing:
    raise ValueError(f'Merged CSV is missing columns: {sorted(missing)}')

if len(df) != 148517:
    print(f'Warning: expected 148,517 rows, found {len(df):,}.')

df = df.drop(columns=['source_file']).copy()
df['label'] = (df['class'].str.lower() != 'normal').astype(int)

categorical_cols = ['protocol_type', 'service', 'flag']
drop_always = ['num_outbound_cmds']
not_features = ['class', 'difficulty_level', 'label']

encoders = {}
for col in categorical_cols:
    encoder = LabelEncoder()
    df[col] = encoder.fit_transform(df[col].astype(str))
    encoders[col] = encoder

df = df.drop(columns=drop_always)
feature_names = [c for c in df.columns if c not in not_features]
X = df[feature_names]
y = df['label'].to_numpy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = RobustScaler().fit(X_train)
X_train_scaled = pd.DataFrame(scaler.transform(X_train), columns=feature_names)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=feature_names)

CONFIRMED_VIF_DROPS = [
    'num_root', 'srv_serror_rate', 'srv_rerror_rate',
    'dst_host_srv_serror_rate', 'flag', 'dst_host_same_srv_rate',
]
kept_features = [
    feature for feature in feature_names if feature not in CONFIRMED_VIF_DROPS
]
dropped_features = CONFIRMED_VIF_DROPS
X_train_final = X_train_scaled[kept_features].to_numpy(dtype=np.float32)
X_test_final = X_test_scaled[kept_features].to_numpy(dtype=np.float32)
print(f'Rows: {len(df):,} | Train: {len(X_train_final):,} | Test: {len(X_test_final):,}')
print(f'Features kept: {len(kept_features)} | Hardcoded VIF drops (from prior analysis): {dropped_features}')

## 4. Preprocessing decisions

This notebook uses the merged NSL-KDD train and test files: 148,517 rows with 41 original features. The original `class` label is collapsed to a binary target: `normal` versus `attack`.

- `num_outbound_cmds` is dropped because it is constant and always zero.
- `protocol_type`, `service`, and `flag` are label-encoded.
- `RobustScaler` is used because it is less sensitive to outliers than `StandardScaler`. **ASSUMPTION:** this is a documented preprocessing choice from the reference pipeline.
- Six features are removed for high multicollinearity using prior VIF analysis with a threshold of 10: `num_root`, `srv_serror_rate`, `srv_rerror_rate`, `dst_host_srv_serror_rate`, `flag`, and `dst_host_same_srv_rate`. **ASSUMPTION:** these VIF results are reused rather than recomputed live in this notebook.

The merged CSV is evaluated with a paper-style protocol: an 80/20 stratified random split created from the merged dataset.

## 5. Build the candidate models

The models match the validated reference pipeline. Bernoulli Naive Bayes is used intentionally instead of GaussianNB because label-encoded categorical values are not continuous Gaussian measurements; the categorical inputs are one-hot encoded inside its pipeline.

In [ ]:
from sklearn.ensemble import RandomForestClassifier, StackingClassifier

categorical_indices = [
    kept_features.index(col) for col in categorical_cols if col in kept_features
]

def make_nb_pipeline():
    return Pipeline([
        ('one_hot_categories', ColumnTransformer(
            transformers=[('categorical', OneHotEncoder(handle_unknown='ignore'), categorical_indices)],
            remainder='passthrough')),
        ('classifier', BernoulliNB()),
    ])

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'K-NN': KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    'Naive Bayes': make_nb_pipeline(),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(
        n_estimators=100, random_state=42, n_jobs=-1,
    ),
    'Stacking (proposed)': StackingClassifier(
        estimators=[
            ('rf', RandomForestClassifier(
                n_estimators=100, random_state=42, n_jobs=-1,
            )),
            ('dt', DecisionTreeClassifier(random_state=42)),
            ('knn', KNeighborsClassifier(n_neighbors=5, n_jobs=-1)),
            ('nb', make_nb_pipeline()),
        ],
        final_estimator=LogisticRegression(max_iter=1000),
        stack_method='predict_proba',
        cv=5,
        n_jobs=-1,
    ),
}

# BernoulliNB is intentional: label-encoded categorical inputs are not Gaussian,
# so the validated pipeline uses one-hot categories with BernoulliNB instead.
stacking_model = models['Stacking (proposed)']
print('Models:', ', '.join(models))

## 6. Evaluate each model on the test set

Each model is fitted on the training split, timed, and evaluated with the same metrics used by `replicate_nslkdd.py`.

In [ ]:
def evaluate_model(name, model):
    started = time.time()
    model.fit(X_train_final, y_train)
    fit_seconds = time.time() - started

    probabilities = model.predict_proba(X_test_final)[:, 1]
    predictions = (probabilities >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, predictions).ravel()

    return {
        'model': name,
        'accuracy': 100 * accuracy_score(y_test, predictions),
        'precision': 100 * precision_score(y_test, predictions, zero_division=0),
        'recall': 100 * recall_score(y_test, predictions, zero_division=0),
        'f1': 100 * f1_score(y_test, predictions, zero_division=0),
        'auc': roc_auc_score(y_test, probabilities),
        'tn': tn,
        'fp': fp,
        'fn': fn,
        'tp': tp,
        'fit_seconds': round(fit_seconds, 1),
    }, probabilities, predictions, confusion_matrix(y_test, predictions)

results_rows = []
model_probabilities = {}
model_predictions = {}
model_confusion_matrices = {}

for name, model in models.items():
    row, probabilities, predictions, matrix = evaluate_model(name, model)
    results_rows.append(row)
    model_probabilities[name] = probabilities
    model_predictions[name] = predictions
    model_confusion_matrices[name] = matrix

results = pd.DataFrame(results_rows)
print(results.round(4).to_string(index=False))

## 7. Five-fold cross-validation

For speed, cross-validation uses a reproducible random subsample capped at 20,000 rows from the training split. These scores are added to the results table.

In [ ]:
from sklearn.model_selection import cross_val_score

cv_rows = []
cv_row_count = min(20000, len(y_train))
rng = np.random.default_rng(0)
cv_indices = rng.choice(len(y_train), size=cv_row_count, replace=False)

for name, model in models.items():
    scores = cross_val_score(
        model,
        X_train_final[cv_indices],
        y_train[cv_indices],
        cv=5,
        scoring='accuracy',
        n_jobs=-1,
    )
    cv_rows.append({
        'model': name,
        'cv_accuracy_mean': 100 * scores.mean(),
        'cv_accuracy_std': 100 * scores.std(),
    })

cv_results = pd.DataFrame(cv_rows)
results = results.drop(columns=['cv_accuracy_mean', 'cv_accuracy_std'], errors='ignore')
results = results.merge(cv_results, on='model', how='left')
print(f'5-fold CV uses {cv_row_count:,} training rows.')
print(results[['model', 'accuracy', 'cv_accuracy_mean', 'cv_accuracy_std']].round(4).to_string(index=False))

## 8. Confusion matrices

The following grid shows actual counts for normal and attack predictions for every model.

In [ ]:
import math
import matplotlib.pyplot as plt

model_names = list(model_confusion_matrices)
columns = min(3, len(model_names))
rows_count = math.ceil(len(model_names) / columns)
fig, axes = plt.subplots(rows_count, columns, figsize=(5 * columns, 4.5 * rows_count), squeeze=False)

for axis, name in zip(axes.ravel(), model_names):
    matrix = model_confusion_matrices[name]
    axis.imshow(matrix, cmap='Blues')
    threshold = matrix.max() / 2
    for row_index in range(2):
        for column_index in range(2):
            axis.text(
                column_index, row_index, f'{matrix[row_index, column_index]:,}',
                ha='center', va='center',
                color='white' if matrix[row_index, column_index] > threshold else 'black',
            )
    axis.set_xticks([0, 1], ['normal', 'attack'])
    axis.set_yticks([0, 1], ['normal', 'attack'])
    axis.set_xlabel('predicted')
    axis.set_ylabel('actual')
    axis.set_title(name)

for axis in axes.ravel()[len(model_names):]:
    axis.axis('off')
fig.suptitle('Confusion matrices for all models')
fig.tight_layout()
plt.show()

## 9. ROC curves

All models are overlaid against a diagonal chance reference.

In [ ]:
from sklearn.metrics import roc_curve
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))
for name, probabilities in model_probabilities.items():
    fpr, tpr, _ = roc_curve(y_test, probabilities)
    plt.plot(fpr, tpr, label=name)
plt.plot([0, 1], [0, 1], 'k--', linewidth=0.8, label='chance')
plt.xlabel('False positive rate')
plt.ylabel('True positive rate')
plt.title('ROC curves for all models')
plt.legend()
plt.tight_layout()
plt.show()

## 10. Comparison with the paper

Baseline accuracy is compared with the paper's reported values using a Â±7 percentage-point tolerance. The proposed stack has no paper baseline.

In [ ]:
PAPER_ACCURACY = {
    'Logistic Regression': 96.72,
    'K-NN': 99.0,
    'Naive Bayes': 89.71,
    'Decision Tree': 97.0,
    'Random Forest': 99.87,
}
TOLERANCE_POINTS = 7.0

print(f'Accuracy comparison with paper (tolerance +/-{TOLERANCE_POINTS:.0f} points):')
for _, row in results.iterrows():
    paper_accuracy = PAPER_ACCURACY.get(row['model'])
    if paper_accuracy is None:
        print(f"{row['model']:<22} proposed model; no paper baseline")
        continue
    difference = row['accuracy'] - paper_accuracy
    status = 'OK' if abs(difference) <= TOLERANCE_POINTS else 'OUT'
    print(f"{row['model']:<22} paper {paper_accuracy:6.2f}  ours {row['accuracy']:6.2f}  "
          f"diff {difference:+6.2f}  [{status}]")

## 11. Closing summary

This notebook uses the merged NSL-KDD CSV and therefore follows the paper-style protocol: the merged train and test data are combined before the stratified 80/20 split. **ASSUMPTION:** these results should be interpreted as merged/paper-style results, not as performance on NSL-KDD's official held-out test split.

The comparison table above reports whether each baseline model is within the paper's Â±7 percentage-point tolerance. In the validated paper-style run, Logistic Regression, K-NN, Bernoulli Naive Bayes, Decision Tree, and Random Forest were all within tolerance. `Stacking (proposed)` is the innovative model, so it has no paper baseline; compare its raw metrics with the baseline rows and the plots above.

# TODO: confirm which protocol this notebook is running if the input CSV or split procedure changes.

## 12. Save and download the PKL

The PKL contains the fitted stacking model, categorical encoders, scaler, retained features, and reused VIF-drop metadata.

In [ ]:
model_bundle = {
    'model_name': 'Stacking (proposed)',
    'model': stacking_model,
    'categorical_encoders': encoders,
    'scaler': scaler,
    'feature_names': feature_names,
    'kept_features': kept_features,
    'dropped_features': dropped_features,
    'categorical_columns': categorical_cols,
    'drop_always': drop_always,
}

pkl_name = 'nslkdd_stacking_proposed.pkl'
with open(pkl_name, 'wb') as handle:
    pickle.dump(model_bundle, handle, protocol=pickle.HIGHEST_PROTOCOL)

print(f'Saved {pkl_name}')
files.download(pkl_name)